In [0]:
%python
from pyspark.sql import functions as F

transformations = {
    '_id': F.col('_id').cast('long'),
    'OCC_DATE': F.timestamp_millis(F.col('OCC_DATE').cast('long')),
}

# These columns can have NSA
transformations |= {
    column: F.when(F.trim(F.upper(F.col(column))) == 'NSA', None) \
        .otherwise(F.col(column))
    for column in [
        'DIVISION',
        'HOOD_158',
        'NEIGHBOURHOOD_158',
    ]
}

# These columns can be
# YES
# NO
# None
# N/R
transformations |= {
    column: F.when(F.trim(F.upper(F.col(column))) == 'YES', True) \
        .when(F.trim(F.upper(F.col(column))) == 'NO', False) \
        .otherwise(None)
    for column in [
        'INJURY_COLLISIONS',
        'FTR_COLLISIONS',
        'PD_COLLISIONS',
        'AUTOMOBILE',
        'MOTORCYCLE',
        'PASSENGER',
        'BICYCLE',
        'PEDESTRIAN',
    ]
}

# FATALITIES can be null or 1+
transformations |= {
    column: F.when(F.col(column).isNull(), 0) \
        .otherwise(F.col(column).cast('int'))
    for column in [
        'OCC_YEAR',
        'OCC_HOUR',
        'FATALITIES',
    ]
}

# These columns can be 0, meaning the coordinates were not specified
transformations |= {
    column: F.when(F.col(column) == '0', None) \
        .otherwise(F.col(column).cast('float'))
    for column in [
        'LONG_WGS84',
        'LAT_WGS84',
    ]
}

df = spark.table('workspace.default.traffic_collisions_bronze') \
    .withColumns(transformations) \
    .drop('geometry') \
    .drop('_rescued_data')

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.traffic_collisions_silver")